# FraSoHome Agents Foundry - Notebook de demo interactiva

Este notebook permite ejecutar paso a paso los módulos Python del repositorio durante la sesión práctica.

La ruta de demo recomendada es:

1. Validar assets del caso, KB y CSVs.
2. Mostrar la base de conocimiento que se cargará en File Search.
3. Perfilar los datos localmente con pandas.
4. Previsualizar la creación de agentes en Foundry.
5. Crear agentes en Foundry, si el entorno está configurado.
6. Ejecutar las tres demos: Knowledge, Data Quality y Orquestador multiagente.

Las celdas que llaman a Azure están protegidas por `RUN_LIVE_FOUNDRY = False` para evitar ejecuciones accidentales.

## 0. Preparación

Antes de abrir el notebook, se recomienda crear el entorno:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
pip install -e .
copy .env.example .env
az login
```

Configura `.env` con:

```text
PROJECT_ENDPOINT=https://<resource>.services.ai.azure.com/api/projects/<project>
MODEL_DEPLOYMENT=<deployment-name>
```

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "frasohome_agents").exists():
    # Useful if the notebook is opened from notebooks/ instead of repo root.
    REPO_ROOT = Path.cwd().parent

sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)

print("Repo root:", REPO_ROOT)
print("Python:", sys.version)

In [ ]:
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

# Safety switch: keep False during rehearsals unless you want to call Microsoft Foundry.
RUN_LIVE_FOUNDRY = False

print("RUN_LIVE_FOUNDRY =", RUN_LIVE_FOUNDRY)
print("PROJECT_ENDPOINT set:", bool(os.getenv("PROJECT_ENDPOINT")))
print("MODEL_DEPLOYMENT:", os.getenv("MODEL_DEPLOYMENT", "<not set>"))

## 1. Cargar configuración y validar assets

Esta sección no llama a Azure. Comprueba que el caso, la KB y los CSV están donde el código espera.

In [ ]:
from frasohome_agents.settings import Settings, ensure_output_dir
from frasohome_agents.assets import assert_assets, knowledge_files, kb_markdown_files, csv_files

settings = Settings.load()
ensure_output_dir(settings)
assert_assets(settings)

print("Case dir:", settings.case_dir)
print("Output dir:", settings.output_dir)
print("Knowledge files:", len(knowledge_files(settings)))
print("KB policy files:", len(kb_markdown_files(settings)))
print("CSV files:", len(csv_files(settings)))

In [ ]:
print("Documentos que se cargarán en File Search:")
for path in knowledge_files(settings):
    print("-", path.relative_to(REPO_ROOT))

print("
CSVs que se cargarán en Code Interpreter:")
for path in csv_files(settings):
    print("-", path.relative_to(REPO_ROOT))

## 2. Mostrar el caso y la KB

Estas celdas ayudan a explicar el storytelling y qué políticas gobiernan las respuestas normativas del agente Knowledge.

In [ ]:
from IPython.display import Markdown, display

case_md = (REPO_ROOT / "case" / "fraso_home_storytelling_foundry.md").read_text(encoding="utf-8")
display(Markdown(case_md[:4000]))

In [ ]:
kb_index = (REPO_ROOT / "case" / "kb" / "README.md").read_text(encoding="utf-8")
display(Markdown(kb_index))

## 3. Perfilado local de datos

Este perfilado usa pandas y no requiere Foundry. Sirve para enseñar el diagnóstico antes de ejecutar Code Interpreter.

In [ ]:
from frasohome_agents.local_data_quality import profile_all

local_report = profile_all(settings)
print("Archivos perfilados:", len(local_report["resumen"]))
print("Reporte JSON:", settings.output_dir / "local_data_quality_report.json")
print("Reporte Markdown:", settings.output_dir / "local_data_quality_report.md")

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(local_report["resumen"])[["archivo", "filas", "columnas", "nulos_totales", "duplicados"]]
summary_df.sort_values(["duplicados", "nulos_totales"], ascending=False)

In [ ]:
from IPython.display import Markdown, display

report_md = (settings.output_dir / "local_data_quality_report.md").read_text(encoding="utf-8")
display(Markdown(report_md))

## 4. Prompts e instrucciones de agentes

Usa estas celdas para enseñar el diseño de instrucciones antes de crear los agentes.

In [ ]:
from frasohome_agents import prompts

print("Knowledge prompt:
", prompts.KNOWLEDGE_PROMPT)
print("
Data Quality prompt:
", prompts.DATA_QUALITY_PROMPT)
print("
Multi-agent prompt:
", prompts.MULTI_AGENT_PROMPT)

In [ ]:
print(prompts.KNOWLEDGE_INSTRUCTIONS[:1600])

## 5. Previsualizar creación de agentes

`dry_run=True` no llama a Azure. Muestra qué agentes, herramientas y archivos se usarán.

In [ ]:
from frasohome_agents.create_agents import create_all_agents

agent_plan = create_all_agents(dry_run=True)
agent_plan.keys()

## 6. Crear agentes en Microsoft Foundry

Activa `RUN_LIVE_FOUNDRY = True` solo cuando:

- `.env` contiene `PROJECT_ENDPOINT` y `MODEL_DEPLOYMENT`.
- Has ejecutado `az login`.
- Tu usuario tiene permisos para crear agentes y cargar archivos.

Esta celda intenta crear:

- `frasohome-knowledge` con File Search.
- `frasohome-data-quality` con Code Interpreter.
- `frasohome-returns` con File Search.
- `frasohome-operations` con File Search.
- `frasohome-storyteller` sin herramientas.

In [ ]:
if RUN_LIVE_FOUNDRY:
    created = create_all_agents(dry_run=False)
    print(created.keys())
else:
    print("Saltado. Cambia RUN_LIVE_FOUNDRY = True para crear agentes en Foundry.")

## 7. Demo 1 - Knowledge

Pregunta canónica:

> Un cliente compró un sofá online, quiere devolverlo en tienda y usó un cupón. ¿Qué pasos debe seguir atención al cliente?

In [ ]:
from frasohome_agents.run_agents import run_knowledge

if RUN_LIVE_FOUNDRY:
    knowledge_answer = run_knowledge()
else:
    print("Saltado. Esta celda llama al agente Knowledge en Foundry.")

## 8. Demo 2 - Data Quality con Code Interpreter

Esta celda llama al agente `frasohome-data-quality`. El agente debe usar Code Interpreter para calcular métricas desde CSVs.

In [ ]:
from frasohome_agents.run_agents import run_data_quality

if RUN_LIVE_FOUNDRY:
    dq_answer = run_data_quality()
else:
    print("Saltado. Esta celda llama a Code Interpreter en Foundry.")

## 9. Demo 3 - Orquestador multiagente en código

Esta celda reproduce el workflow visual desde Python:

1. Knowledge aporta contexto normativo.
2. Data Quality aporta evidencia de datos.
3. Returns interpreta devoluciones.
4. Operations interpreta causas operativas.
5. Storyteller sintetiza la acción de 7 días.

In [ ]:
from frasohome_agents.orchestrator import run_orchestrator

if RUN_LIVE_FOUNDRY:
    orchestrator_result = run_orchestrator()
else:
    print("Saltado. Esta celda invoca varios agentes en Foundry.")

## 10. Revisar salidas generadas

Las ejecuciones guardan resultados en `outputs/` para poder enseñarlos o compararlos.

In [ ]:
for path in sorted((REPO_ROOT / "outputs").glob("*")):
    print(path.name, path.stat().st_size, "bytes")

## 11. Cierre de sesión

Puntos para comentar al final:

- File Search aporta grounding documental sobre la KB.
- Code Interpreter calcula métricas y evita inventar cifras.
- El orquestador separa contexto, cálculo, interpretación y síntesis.
- La salida final debe citar fuentes, declarar riesgos y proponer una acción medible.
- Las trazas y tool calls son parte de la demo, no un detalle técnico secundario.